# Human-in-the-Loop with `interrupt`

**What you will build:** a graph that **pauses mid-run** to wait for a person, then **resumes** exactly where it left off. In 0.5 we did this with `input()`, which only works in a live cell. LangGraph's `interrupt` does it properly: the run stops, its full state is checkpointed, and you can resume later — even from a web request. Maps to chapter 04d of n8n.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/24_human_in_the_loop.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — LangChain + LangGraph on OpenRouter. Run once.
!pip install -q "langchain>=1.3,<2.0" "langgraph>=1.2,<2.0" "langchain-openai>=1.3,<2.0"

import os, getpass
from langchain_openai import ChatOpenAI

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"
llm = ChatOpenAI(model=MODEL_NAME, base_url="https://openrouter.ai/api/v1",
                 api_key=os.environ["OPENROUTER_API_KEY"])
print("Ready:", MODEL_NAME)

## A draft-approve-send graph

The agent drafts an email, then **stops** for approval before the irreversible send. `interrupt()` is what pauses it. Human-in-the-loop **requires a checkpointer** (2.3) — the state has to be saved in order to resume.

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

class State(TypedDict):
    request: str
    draft: str
    decision: str
    result: str

def write_draft(state: State) -> dict:
    r = llm.invoke([{"role": "system", "content": "Draft a short, professional email. Output only the body."},
                    {"role": "user", "content": state["request"]}])
    return {"draft": r.content}

def human_review(state: State) -> dict:
    # This call PAUSES the graph and hands the payload back to the caller.
    decision = interrupt({"draft": state["draft"], "question": "Approve this email? (yes/no)"})
    return {"decision": decision}          # <- filled with the resume value when we continue

def send(state: State) -> dict:
    if "yes" in state["decision"].lower():
        return {"result": "Email sent."}   # real send would go here
    return {"result": "Cancelled - nothing was sent."}

builder = StateGraph(State)
builder.add_node("write_draft", write_draft)
builder.add_node("human_review", human_review)
builder.add_node("send", send)
builder.add_edge(START, "write_draft")
builder.add_edge("write_draft", "human_review")
builder.add_edge("human_review", "send")
builder.add_edge("send", END)

graph = builder.compile(checkpointer=InMemorySaver())   # checkpointer is REQUIRED for interrupt

In [ ]:
from IPython.display import Image, display
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print("Mermaid image unavailable, showing text instead:\n")
    print(graph.get_graph().draw_mermaid())

## Run until the pause

Invoke it. The graph runs `write_draft`, then stops at `human_review`; the return value carries the interrupt payload.

In [ ]:
cfg = {"configurable": {"thread_id": "email-1"}}

paused = graph.invoke({"request": "Ask the vendor to move Tuesday's demo to Thursday."}, config=cfg)
print("PAUSED. The graph is waiting for a human.")
print("interrupt payload:", paused["__interrupt__"][0].value)

The graph is frozen at `human_review`, its state safely checkpointed under `thread_id='email-1'`. Nothing was sent. Now a human decides, and we **resume** by invoking with a `Command(resume=...)` — the value becomes the return of `interrupt()`.

In [ ]:
# A person reviews the draft above and approves. Resume the SAME thread:
final = graph.invoke(Command(resume="yes"), config=cfg)
print("result:", final["result"])

The run picked up exactly where it stopped — `human_review` returned `"yes"`, and `send` ran. Because the state is checkpointed, the pause can last milliseconds or days, and resume from a different process entirely. That's real human-in-the-loop, and it's what the `input()` version in 0.5 could never do.

```{tip}
Put the interrupt right **before** the irreversible action, not around the whole graph — same principle as 0.5. Let the model draft freely; gate only the send.
```

```{note}
You built human-in-the-loop from graph primitives so you understand it. In production, **LangChain 1.0 ships prebuilt middleware** — including human-in-the-loop, conversation summarization, and PII/content moderation — that wraps these patterns for you. Build it once by hand to learn it, then reach for middleware: the same philosophy as the rest of this course. And it isn't only LangGraph — **PydanticAI 2.x** also ships tool-approval flags for human-in-the-loop and durable-execution integrations (Temporal, DBOS), so you can pause-and-resume there too.
```

**What's next:** in **2.5** we use LangGraph's native cycles for the **reflection** pattern — a generate/critique loop drawn as a real loop.